In [3]:
# -*- coding: utf-8 -*-
"""
backend9_notebook_planned_and_nonop_summary.ipynb (single-run, notebook)

목표 (prod_day=20260129, shift_type=night 고정)
1) g_production_film.planned_time -> 계획 정지 시간 총합 df (1행)
2) g_production_film.fct_non_operation_time + vision_non_operation_time
   -> planned_time 구간 제외(겹치는 부분만 잘라서 제외)
   -> station별 비가동 시간 df (1행, 한글 컬럼명)

사양 근거:
- 주/야간 기준: day=08:30:00~20:29:59, night=20:30:00~D+1 08:29:59
- planned_time: end_day/from_time/to_time/reason (HH:MI:SS)
- non_operation: from/to 소수 초(.xx) 가능 -> .5에서 반올림(표준 half-up)하여 HH:MI:SS 정규화
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo
from typing import List, Tuple, Dict

import pandas as pd
from sqlalchemy import create_engine, text

# =========================
# 0) 고정 입력
# =========================
KST = ZoneInfo("Asia/Seoul")

TARGET_PROD_DAY = "20260130"
TARGET_SHIFT = "day"  # "day" or "night"

# =========================
# 1) DB 설정 (사양서)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

SCHEMA = "g_production_film"
PLANNED_TABLE = "planned_time"
FCT_NONOP_TABLE = "fct_non_operation_time"
VISION_NONOP_TABLE = "vision_non_operation_time"

STATIONS = ["FCT1", "FCT2", "FCT3", "FCT4", "Vision1", "Vision2"]

# =========================
# 2) 유틸
# =========================
def make_engine():
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    # notebook: 과도한 풀 방지
    return create_engine(url, pool_size=1, max_overflow=0, pool_pre_ping=True)

def parse_yyyymmdd(s: str) -> date:
    return date(int(s[0:4]), int(s[4:6]), int(s[6:8]))

def fmt_yyyymmdd(d: date) -> str:
    return d.strftime("%Y%m%d")

def parse_hms(s: str) -> time:
    # "HH:MI:SS"
    hh, mm, ss = s.split(":")
    return time(int(hh), int(mm), int(ss))

def parse_hms_round_half_up(s: str) -> time:
    """
    "HH:MI:SS.xx" 또는 "HH:MI:SS"
    - .5에서 표준 반올림(half-up)하여 초 단위 time 반환
    - Python round()는 banker's rounding이라 사용 금지
    """
    # 정수 초만 있으면 그대로
    if "." not in s:
        return parse_hms(s)

    base, frac = s.split(".", 1)
    t0 = parse_hms(base)
    # frac은 "45" 같은 형태(소수점 이하). 길이는 가변일 수 있음.
    # float 오차 방지: "0." + frac 을 Decimal로 처리해도 되지만,
    # 여기서는 문자열 기반으로 microsecond를 구성한 뒤 +500ms 반올림.
    # 예: .45 => 450ms, .5 => 500ms, .32 => 320ms
    frac_digits = "".join(ch for ch in frac if ch.isdigit())
    if frac_digits == "":
        ms = 0
    else:
        # 최대 3자리(ms)까지만 의미 있게 취급(2자리면 centisecond)
        if len(frac_digits) >= 3:
            ms = int(frac_digits[:3])
        elif len(frac_digits) == 2:
            ms = int(frac_digits) * 10
        else:  # len == 1
            ms = int(frac_digits) * 100

    total_ms = (t0.hour * 3600 + t0.minute * 60 + t0.second) * 1000 + ms
    # half-up: +500ms 후 //1000
    total_sec = (total_ms + 500) // 1000

    # 24h wrap (안전)
    total_sec = total_sec % (24 * 3600)
    hh = total_sec // 3600
    mm = (total_sec % 3600) // 60
    ss = total_sec % 60
    return time(int(hh), int(mm), int(ss))

def shift_window(prod_day: str, shift: str) -> Tuple[datetime, datetime]:
    """
    return (start_dt_inclusive, end_dt_exclusive) in KST
    - day: 08:30:00 ~ 20:30:00(exclusive)
    - night: 20:30:00 ~ D+1 08:30:00(exclusive)
    """
    d = parse_yyyymmdd(prod_day)
    if shift == "day":
        start = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
        end_excl = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
    elif shift == "night":
        start = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
        d2 = d + timedelta(days=1)
        end_excl = datetime(d2.year, d2.month, d2.day, 8, 30, 0, tzinfo=KST)
    else:
        raise ValueError("shift must be 'day' or 'night'")
    return start, end_excl

def classify_prod_shift(end_day: str, t: time) -> Tuple[str, str]:
    """
    사용자 정의 규칙(너가 준 #5 반영):
    - day: 08:30:00~20:29:59  => prod_day=end_day
    - night:
        * 20:30:00~23:59:59 => prod_day=end_day
        * 00:00:00~08:29:59 => prod_day=end_day - 1day
    """
    day_start = time(8, 30, 0)
    night_start = time(20, 30, 0)
    night_end = time(8, 29, 59)

    if day_start <= t <= time(20, 29, 59):
        return end_day, "day"

    # night
    if t >= night_start:
        return end_day, "night"

    # 00:00:00 ~ 08:29:59
    if t <= night_end:
        d = parse_yyyymmdd(end_day) - timedelta(days=1)
        return fmt_yyyymmdd(d), "night"

    # (여기 오면 사실상 공백인데, 안전)
    return end_day, "unknown"

def seconds_to_kor(sec: int) -> str:
    if sec <= 0:
        return "0초"
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    parts = []
    if h:
        parts.append(f"{h}시간")
    if m:
        parts.append(f"{m}분")
    if s:
        parts.append(f"{s}초")
    return " ".join(parts) if parts else "0초"

@dataclass(frozen=True)
class Seg:
    start: datetime
    end: datetime  # exclusive

def overlap(a: Seg, b: Seg) -> Seg | None:
    s = max(a.start, b.start)
    e = min(a.end, b.end)
    if s < e:
        return Seg(s, e)
    return None

def merge_segments(segs: List[Seg]) -> List[Seg]:
    if not segs:
        return []
    segs_sorted = sorted(segs, key=lambda x: x.start)
    merged = [segs_sorted[0]]
    for cur in segs_sorted[1:]:
        last = merged[-1]
        if cur.start <= last.end:  # 접하거나 겹치면 합침 (exclusive 기준)
            merged[-1] = Seg(last.start, max(last.end, cur.end))
        else:
            merged.append(cur)
    return merged

def subtract_planned_seconds(interval: Seg, planned_union: List[Seg]) -> int:
    """
    interval에서 planned_union(이미 merge된 세그먼트들) 겹치는 부분만 빼고 남은 초 반환
    """
    total = int((interval.end - interval.start).total_seconds())
    if total <= 0:
        return 0

    ov = []
    for p in planned_union:
        inter = overlap(interval, p)
        if inter:
            ov.append(inter)
    ov = merge_segments(ov)
    ov_sec = sum(int((x.end - x.start).total_seconds()) for x in ov)
    remain = total - ov_sec
    return max(0, remain)

# =========================
# 3) 데이터 로드 (필요 end_day 범위만)
# =========================
window_start, window_end_excl = shift_window(TARGET_PROD_DAY, TARGET_SHIFT)
end_day_1 = TARGET_PROD_DAY
end_day_2 = fmt_yyyymmdd(parse_yyyymmdd(TARGET_PROD_DAY) + timedelta(days=1))

engine = make_engine()

# planned_time: end_day in {D, D+1} 가져온 뒤 파이썬에서 window/shift로 필터
q_planned = text(f"""
    SELECT end_day, from_time, to_time, reason
    FROM {SCHEMA}.{PLANNED_TABLE}
    WHERE end_day IN :days
""")

# non_operation: fct + vision 각각 조회
q_nonop = lambda table: text(f"""
    SELECT end_day, station, from_time, to_time
    FROM {SCHEMA}.{table}
    WHERE end_day IN :days
""")

with engine.connect() as conn:
    planned_raw = pd.read_sql(q_planned, conn, params={"days": tuple([end_day_1, end_day_2])})
    fct_raw = pd.read_sql(q_nonop(FCT_NONOP_TABLE), conn, params={"days": tuple([end_day_1, end_day_2])})
    vision_raw = pd.read_sql(q_nonop(VISION_NONOP_TABLE), conn, params={"days": tuple([end_day_1, end_day_2])})

nonop_raw = pd.concat([fct_raw, vision_raw], ignore_index=True)

# =========================
# 4) planned_time: 구간 분할 + TARGET window만 남기기
# =========================
planned_segs_target: List[Seg] = []
planned_total_sec = 0

# 분할 경계(한 "calendar day" 안에서만 쪼갠다고 가정: from<=to, 자정跨越 없음)
CUTS = [time(8, 30, 0), time(20, 30, 0)]

for _, r in planned_raw.iterrows():
    end_day = str(r["end_day"])
    ft = parse_hms(str(r["from_time"]))
    tt = parse_hms(str(r["to_time"]))

    d = parse_yyyymmdd(end_day)
    start_dt = datetime(d.year, d.month, d.day, ft.hour, ft.minute, ft.second, tzinfo=KST)
    end_dt = datetime(d.year, d.month, d.day, tt.hour, tt.minute, tt.second, tzinfo=KST)

    if end_dt <= start_dt:
        continue  # 사양상 없다고 했으니 skip (안전)

    # cutpoints 생성
    cut_dts = [start_dt, end_dt]
    for c in CUTS:
        cdt = datetime(d.year, d.month, d.day, c.hour, c.minute, c.second, tzinfo=KST)
        if start_dt < cdt < end_dt:
            cut_dts.append(cdt)
    cut_dts = sorted(set(cut_dts))

    # segment로 쪼개기
    for sdt, edt in zip(cut_dts[:-1], cut_dts[1:]):
        # segment의 shift/prod_day는 segment 시작 시각 기준으로 분류(쪼갰기 때문에 안전)
        prod_day_seg, shift_seg = classify_prod_shift(end_day, sdt.timetz().replace(tzinfo=None))

        if prod_day_seg != TARGET_PROD_DAY or shift_seg != TARGET_SHIFT:
            continue

        seg = Seg(sdt, edt)
        # window 교차로 안전하게 자르기
        inter = overlap(seg, Seg(window_start, window_end_excl))
        if not inter:
            continue

        sec = int((inter.end - inter.start).total_seconds())
        if sec <= 0:
            continue

        planned_segs_target.append(inter)
        planned_total_sec += sec

# planned 구간은 겹칠 수 있으니(중복 row 등) union으로 정리 후 재합산(중복계산 방지)
planned_union = merge_segments(planned_segs_target)
planned_total_sec = sum(int((x.end - x.start).total_seconds()) for x in planned_union)

updated_at = datetime.now(tz=KST)

df_planned_summary = pd.DataFrame([{
    "prod_day": TARGET_PROD_DAY,
    "shift_type": TARGET_SHIFT,
    "Total 계획 정지 시간": seconds_to_kor(int(planned_total_sec)),
    "total_planned_time": int(planned_total_sec),
    "updated_at": pd.Timestamp(updated_at),
}])

# =========================
# 5) non_operation: window 자르기 + planned_time 겹침 부분만 제거 + station별 합산
# =========================
station_sec: Dict[str, int] = {s: 0 for s in STATIONS}

for _, r in nonop_raw.iterrows():
    end_day = str(r["end_day"])
    station = str(r.get("station", "")).strip()

    if station not in station_sec:
        continue

    ft = parse_hms_round_half_up(str(r["from_time"]))
    tt = parse_hms_round_half_up(str(r["to_time"]))

    d = parse_yyyymmdd(end_day)
    start_dt = datetime(d.year, d.month, d.day, ft.hour, ft.minute, ft.second, tzinfo=KST)
    end_dt = datetime(d.year, d.month, d.day, tt.hour, tt.minute, tt.second, tzinfo=KST)

    # 혹시 자정跨越가 들어오면(안전) 다음날로 보정
    if end_dt <= start_dt:
        end_dt = end_dt + timedelta(days=1)

    # (prod_day,shift) 분류는 "시작 시각" 기준으로 하되, window로 잘라서 TARGET만 남김
    prod_day_seg, shift_seg = classify_prod_shift(end_day, start_dt.timetz().replace(tzinfo=None))
    if prod_day_seg != TARGET_PROD_DAY or shift_seg != TARGET_SHIFT:
        # 일부 interval이 boundary를 걸치면 start 기준만으로는 놓칠 수 있음
        # => 먼저 window 교차로 자른 뒤, 그 교차 구간의 시작 시각으로 재분류
        inter0 = overlap(Seg(start_dt, end_dt), Seg(window_start, window_end_excl))
        if not inter0:
            continue
        prod_day2, shift2 = classify_prod_shift(inter0.start.strftime("%Y%m%d"), inter0.start.timetz().replace(tzinfo=None))
        if prod_day2 != TARGET_PROD_DAY or shift2 != TARGET_SHIFT:
            continue
        interval = inter0
    else:
        interval = overlap(Seg(start_dt, end_dt), Seg(window_start, window_end_excl))
        if not interval:
            continue

    # planned_time 구간(전체 공통)과 겹치는 부분만 잘라서 제외
    remain_sec = subtract_planned_seconds(interval, planned_union)
    if remain_sec <= 0:
        continue

    station_sec[station] += remain_sec

vision_total_sec = station_sec["Vision1"] + station_sec["Vision2"]

df_nonop_summary = pd.DataFrame([{
    "prod_day": TARGET_PROD_DAY,
    "shift_type": TARGET_SHIFT,
    "비가동 FCT1": seconds_to_kor(station_sec["FCT1"]),
    "비가동 FCT2": seconds_to_kor(station_sec["FCT2"]),
    "비가동 FCT3": seconds_to_kor(station_sec["FCT3"]),
    "비가동 FCT4": seconds_to_kor(station_sec["FCT4"]),
    "비가동 Vision1": seconds_to_kor(station_sec["Vision1"]),
    "비가동 Vision2": seconds_to_kor(station_sec["Vision2"]),
    "Total Vision 비가동 시간": seconds_to_kor(vision_total_sec),
    "total_vision_non_time": int(vision_total_sec),
    "updated_at": pd.Timestamp(updated_at),
}])

# =========================
# 6) 출력
# =========================
print("=== 계획 정지 시간 요약 (1행) ===")
display(df_planned_summary)

print("\n=== 비가동 시간 요약 (1행) ===")
display(df_nonop_summary)

=== 계획 정지 시간 요약 (1행) ===


,prod_day,shift_type,Total 계획 정지 시간,total_planned_time,updated_at
0,20260130,day,1시간 10초,3610,2026-01-30 16:40:45.702147+09:00



=== 비가동 시간 요약 (1행) ===


,prod_day,shift_type,비가동 FCT1,비가동 FCT2,비가동 FCT3,비가동 FCT4,비가동 Vision1,비가동 Vision2,Total Vision 비가동 시간,total_vision_non_time,updated_at
0,20260130,day,29분 11초,17분 1초,30분 3초,23분 6초,3시간 12분 1초,3시간 4분 22초,6시간 16분 23초,22583,2026-01-30 16:40:45.702147+09:00


In [4]:
# =========================
# [추가 셀] 결과 DF를 i_daily_report에 저장(UPSERT)
# - shift_type=day  -> *_day 테이블
# - shift_type=night-> *_night 테이블
# =========================

from sqlalchemy import text

DST_SCHEMA = "i_daily_report"
T_PLANNED_DAY = "total_planned_time_day"
T_PLANNED_NIGHT = "total_planned_time_night"
T_NONOP_DAY = "total_non_time_day"
T_NONOP_NIGHT = "total_non_time_night"


def _ensure_dst_tables(conn):
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {DST_SCHEMA}"))

    # planned (day)
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {DST_SCHEMA}.{T_PLANNED_DAY} (
            prod_day TEXT NOT NULL,
            shift_type TEXT NOT NULL,
            "Total 계획 정지 시간" TEXT NOT NULL,
            total_planned_time INTEGER NOT NULL,
            updated_at TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (prod_day, shift_type)
        )
    """))

    # planned (night)
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {DST_SCHEMA}.{T_PLANNED_NIGHT} (
            prod_day TEXT NOT NULL,
            shift_type TEXT NOT NULL,
            "Total 계획 정지 시간" TEXT NOT NULL,
            total_planned_time INTEGER NOT NULL,
            updated_at TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (prod_day, shift_type)
        )
    """))

    # nonop (day)
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {DST_SCHEMA}.{T_NONOP_DAY} (
            prod_day TEXT NOT NULL,
            shift_type TEXT NOT NULL,
            "비가동 FCT1" TEXT NOT NULL,
            "비가동 FCT2" TEXT NOT NULL,
            "비가동 FCT3" TEXT NOT NULL,
            "비가동 FCT4" TEXT NOT NULL,
            "비가동 Vision1" TEXT NOT NULL,
            "비가동 Vision2" TEXT NOT NULL,
            "Total Vision 비가동 시간" TEXT NOT NULL,
            total_vision_non_time INTEGER NOT NULL,
            updated_at TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (prod_day, shift_type)
        )
    """))

    # nonop (night)
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {DST_SCHEMA}.{T_NONOP_NIGHT} (
            prod_day TEXT NOT NULL,
            shift_type TEXT NOT NULL,
            "비가동 FCT1" TEXT NOT NULL,
            "비가동 FCT2" TEXT NOT NULL,
            "비가동 FCT3" TEXT NOT NULL,
            "비가동 FCT4" TEXT NOT NULL,
            "비가동 Vision1" TEXT NOT NULL,
            "비가동 Vision2" TEXT NOT NULL,
            "Total Vision 비가동 시간" TEXT NOT NULL,
            total_vision_non_time INTEGER NOT NULL,
            updated_at TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (prod_day, shift_type)
        )
    """))

    conn.commit()


def _upsert_planned(conn, df_one_row: pd.DataFrame):
    row = df_one_row.iloc[0].to_dict()
    shift = row["shift_type"]
    dst = T_PLANNED_DAY if shift == "day" else T_PLANNED_NIGHT

    q = text(f"""
        INSERT INTO {DST_SCHEMA}.{dst}
            (prod_day, shift_type, "Total 계획 정지 시간", total_planned_time, updated_at)
        VALUES
            (:prod_day, :shift_type, :kor, :sec, :updated_at)
        ON CONFLICT (prod_day, shift_type)
        DO UPDATE SET
            "Total 계획 정지 시간" = EXCLUDED."Total 계획 정지 시간",
            total_planned_time = EXCLUDED.total_planned_time,
            updated_at = EXCLUDED.updated_at
    """)
    conn.execute(q, {
        "prod_day": str(row["prod_day"]),
        "shift_type": str(row["shift_type"]),
        "kor": str(row["Total 계획 정지 시간"]),
        "sec": int(row["total_planned_time"]),
        "updated_at": row["updated_at"],
    })
    conn.commit()


def _upsert_nonop(conn, df_one_row: pd.DataFrame):
    row = df_one_row.iloc[0].to_dict()
    shift = row["shift_type"]
    dst = T_NONOP_DAY if shift == "day" else T_NONOP_NIGHT

    q = text(f"""
        INSERT INTO {DST_SCHEMA}.{dst}
            (prod_day, shift_type,
             "비가동 FCT1","비가동 FCT2","비가동 FCT3","비가동 FCT4",
             "비가동 Vision1","비가동 Vision2",
             "Total Vision 비가동 시간", total_vision_non_time, updated_at)
        VALUES
            (:prod_day, :shift_type,
             :fct1,:fct2,:fct3,:fct4,
             :v1,:v2,
             :vtotal_kor,:vtotal,:updated_at)
        ON CONFLICT (prod_day, shift_type)
        DO UPDATE SET
            "비가동 FCT1" = EXCLUDED."비가동 FCT1",
            "비가동 FCT2" = EXCLUDED."비가동 FCT2",
            "비가동 FCT3" = EXCLUDED."비가동 FCT3",
            "비가동 FCT4" = EXCLUDED."비가동 FCT4",
            "비가동 Vision1" = EXCLUDED."비가동 Vision1",
            "비가동 Vision2" = EXCLUDED."비가동 Vision2",
            "Total Vision 비가동 시간" = EXCLUDED."Total Vision 비가동 시간",
            total_vision_non_time = EXCLUDED.total_vision_non_time,
            updated_at = EXCLUDED.updated_at
    """)

    conn.execute(q, {
        "prod_day": str(row["prod_day"]),
        "shift_type": str(row["shift_type"]),
        "fct1": str(row["비가동 FCT1"]),
        "fct2": str(row["비가동 FCT2"]),
        "fct3": str(row["비가동 FCT3"]),
        "fct4": str(row["비가동 FCT4"]),
        "v1": str(row["비가동 Vision1"]),
        "v2": str(row["비가동 Vision2"]),
        "vtotal_kor": str(row["Total Vision 비가동 시간"]),
        "vtotal": int(row["total_vision_non_time"]),
        "updated_at": row["updated_at"],
    })
    conn.commit()


# 실행: 기존 engine 재사용
with engine.connect() as conn:
    _ensure_dst_tables(conn)

    # planned 저장
    _upsert_planned(conn, df_planned_summary)

    # nonop 저장
    _upsert_nonop(conn, df_nonop_summary)

print("✅ 저장 완료:",
      f"{DST_SCHEMA}.{T_PLANNED_DAY}/{T_PLANNED_NIGHT},",
      f"{DST_SCHEMA}.{T_NONOP_DAY}/{T_NONOP_NIGHT}")

✅ 저장 완료: i_daily_report.total_planned_time_day/total_planned_time_night, i_daily_report.total_non_time_day/total_non_time_night
